# CropSeg — Dataset statistics and model facts

This notebook reproduces the dataset statistics and model facts quoted in the README and the technical report. It does **not** train anything and does not need a GPU.

1. Number of images per split and recording date
2. Class distribution (soil / crop / weed) over all label masks, per split and per date
3. What is stored inside `best_model.pt` (epoch, validation loss, validation mIoU)
4. Model size (number of parameters), checked against the trained checkpoint

Results are saved to `MyDrive/CropSeg/evaluation/dataset_and_model_stats.json`.

## 0. Setup

In [1]:
# Mount Google Drive (dataset and checkpoint live there)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install the segmentation library (PyTorch and OpenCV are preinstalled on Colab)
import subprocess
subprocess.check_call(["pip", "install", "-q", "segmentation-models-pytorch"])

0

In [3]:
from pathlib import Path

CHECKPOINT_PATH = Path("/content/drive/MyDrive/CropSeg/checkpoints/best_model.pt")
DATA_ROOT = Path("/content/drive/MyDrive/CropSeg/data/phenobench")
OUTPUT_DIR = Path("/content/drive/MyDrive/CropSeg/evaluation")


def find_phenobench_dir(root):
    # The folder that contains train/ and val/ (the zip may add extra nesting)
    for cand in [root, *sorted(root.glob("*")), *sorted(root.glob("*/*"))]:
        if (cand / "train" / "images").is_dir() and (cand / "val" / "images").is_dir():
            return cand
    raise FileNotFoundError(f"No folder with train/images and val/images under {root}")


PHENOBENCH_DIR = find_phenobench_dir(DATA_ROOT)
train_images = sorted((PHENOBENCH_DIR / "train" / "images").glob("*.png"))
val_images = sorted((PHENOBENCH_DIR / "val" / "images").glob("*.png"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PhenoBench folder: {PHENOBENCH_DIR}")
print(f"Train images: {len(train_images)} (expected 1407)")
print(f"Val images:   {len(val_images)} (expected 772)")
assert CHECKPOINT_PATH.exists(), "best_model.pt not found - check CHECKPOINT_PATH"
results = {}

PhenoBench folder: /content/drive/MyDrive/CropSeg/data/phenobench/PhenoBench
Train images: 1407 (expected 1407)
Val images:   772 (expected 772)


## 1. Images per split and recording date

The filename starts with the recording date (2020), e.g. `05-15_00028_P0030852.png`.

In [4]:
from collections import Counter

results["images_per_date"] = {}
for split, paths in (("train", train_images), ("val", val_images)):
    per_date = dict(sorted(Counter(p.name[:5] for p in paths).items()))
    results["images_per_date"][split] = per_date
    print(f"{split}: {len(paths)} images, per date: {per_date}")

train: 1407 images, per date: {'05-15': 613, '05-26': 396, '06-05': 398}
val: 772 images, per date: {'05-15': 399, '05-26': 170, '06-05': 203}


## 2. Class distribution over all label masks

PhenoBench labels are merged with the official rule: partial crop (3) → crop (1), partial weed (4) → weed (2). Reads about 2,200 masks from Drive, so it takes a few minutes.

In [5]:
import cv2
import numpy as np
from collections import defaultdict
from tqdm.auto import tqdm

LUT = np.zeros(65536, np.uint8); LUT[[1, 3]] = 1; LUT[[2, 4]] = 2   # 16-bit masks; official 3->crop, 4->weed
results["class_distribution"] = {}
for split, paths in (("train", train_images), ("val", val_images)):
    counts = defaultdict(lambda: np.zeros(3, np.int64))
    for p in tqdm(paths, desc=split):
        m = LUT[cv2.imread(str(p.parent.parent / "semantics" / p.name), cv2.IMREAD_UNCHANGED)]
        c = np.bincount(m.ravel(), minlength=3)
        counts[p.name[:5]] += c
        counts["ALL"] += c
    for k, c in sorted(counts.items()):
        pct = c / c.sum() * 100
        print(f"{split} {k:6s} soil {pct[0]:5.2f}%  crop {pct[1]:5.2f}%  weed {pct[2]:5.2f}%  soil:weed {c[0]/c[2]:.0f}:1")
        results["class_distribution"][f"{split}_{k}"] = {
            "pixels": c.tolist(), "soil_pct": pct[0], "crop_pct": pct[1], "weed_pct": pct[2],
            "soil_to_weed": c[0] / c[2]}

train:   0%|          | 0/1407 [00:00<?, ?it/s]

train 05-15  soil 96.70%  crop  3.12%  weed  0.19%  soil:weed 517:1
train 05-26  soil 88.14%  crop 11.50%  weed  0.36%  soil:weed 245:1
train 06-05  soil 73.25%  crop 25.64%  weed  1.11%  soil:weed 66:1
train ALL    soil 87.66%  crop 11.85%  weed  0.50%  soil:weed 176:1


val:   0%|          | 0/772 [00:00<?, ?it/s]

val 05-15  soil 96.83%  crop  3.04%  weed  0.13%  soil:weed 744:1
val 05-26  soil 89.84%  crop  9.79%  weed  0.37%  soil:weed 244:1
val 06-05  soil 76.40%  crop 22.15%  weed  1.45%  soil:weed 53:1
val ALL    soil 89.92%  crop  9.55%  weed  0.53%  soil:weed 170:1


## 3. What is stored inside `best_model.pt`

In [6]:
import torch

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
print("keys:", sorted(ckpt.keys()))
print(f"epoch (0-based): {ckpt['epoch']} -> printed as 'Epoch {ckpt['epoch'] + 1}' in the training log")
print("val_loss:", ckpt["val_loss"])
print("val_miou (batch-averaged):", ckpt["val_miou"])
print("config:", ckpt["config"])
results["checkpoint"] = {"epoch_0_based": int(ckpt["epoch"]), "val_loss": float(ckpt["val_loss"]),
                         "val_miou_batch_avg": float(ckpt["val_miou"]), "config": ckpt["config"]}

keys: ['config', 'epoch', 'model_state_dict', 'optimizer_state_dict', 'val_loss', 'val_miou']
epoch (0-based): 38 -> printed as 'Epoch 39' in the training log
val_loss: 0.14366250899956398
val_miou (batch-averaged): 0.8224810558461848
config: {'encoder_name': 'resnet34', 'num_classes': 3, 'image_size': 512}


## 4. Model size

Builds the same U-Net as `training.ipynb`, loads the trained weights into it (this only succeeds if the architecture matches exactly), and counts the parameters.

In [7]:
import segmentation_models_pytorch as smp

model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=3)
print("Loading trained weights:", model.load_state_dict(ckpt["model_state_dict"]))


def count(module):
    return sum(p.numel() for p in module.parameters())


results["parameters"] = {"total": count(model), "encoder": count(model.encoder),
                         "decoder": count(model.decoder), "segmentation_head": count(model.segmentation_head)}
for part, n in results["parameters"].items():
    print(f"{part:18s} {n:>12,}")

Loading trained weights: <All keys matched successfully>
total                24,436,659
encoder              21,284,672
decoder               3,151,552
segmentation_head           435


## 5. Save

In [8]:
import json

out = OUTPUT_DIR / "dataset_and_model_stats.json"
with open(out, "w") as f:
    json.dump(results, f, indent=2, default=float)
print(f"Saved {out}")

Saved /content/drive/MyDrive/CropSeg/evaluation/dataset_and_model_stats.json
